# Extract

In [2]:
# importing pandas
import pandas as pd

In [3]:
# importing customer dataset csv
customers = pd.read_csv("olist_customers_dataset.csv")

In [4]:
# checking success of import
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [5]:
# importing products excel file
products = pd.read_excel("olist_products_dataset.xlsx")

In [6]:
# checking success of import
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [7]:
import requests

In [8]:
# extracting engagement
response = requests.get('https://fakestoreapi.com/carts')
engagement = pd.json_normalize(response.json())

In [9]:
# importing os to create directory
import os

In [10]:
# creating directory
os.makedirs("etl-project", exist_ok=True)

In [11]:
os.makedirs("etl-project/data/raw", exist_ok=True)

In [12]:
os.makedirs("etl-project/data/clean", exist_ok=True)

In [13]:
os.makedirs("etl-project/data/final", exist_ok=True)

In [14]:
os.makedirs("etl-project/extract", exist_ok=True)

In [15]:
os.makedirs("etl-project/transform", exist_ok=True)

In [16]:
os.makedirs("etl-project/load", exist_ok=True)

In [17]:
os.makedirs("etl-project/docs", exist_ok=True)

In [18]:
customers.to_csv("etl-project/data/raw/customers.csv")

In [19]:
products.to_excel("etl-project/data/raw/products.xlsx")

# Transform

## Standardize Column Names

In [20]:
customers.columns = customers.columns.str.lower().str.replace(" ", "_")
products.columns = products.columns.str.lower().str.replace(" ", "_")
engagement.columns = engagement.columns.str.lower().str.replace(" ", "_")

## Fix Data Types

In [21]:
customers["customer_id"] = customers["customer_id"].astype(str)
products["product_id"] = products["product_id"].astype(str)

In [22]:
engagement["date"] = pd.to_datetime(engagement["date"])

## Handle Missing Values

In [23]:
customers.fillna("unknown", inplace=True)
products.fillna(products.mean(numeric_only=True), inplace=True)

## Convert Dates

In [24]:
engagement["date"] = pd.to_datetime(engagement["date"])

## Drop Duplicates

In [25]:
customers.drop_duplicates(subset='customer_id', inplace=True)
products.drop_duplicates(subset='product_id', inplace=True)

## Save Cleaned Data

In [26]:
customers.to_csv("etl-project/data/clean/customers_clean.csv", index=False)
products.to_csv("etl-project/data/clean/products_clean.csv", index=False)
engagement.to_csv("etl-project/data/clean/engagement_clean.csv", index=False)

# Load

## Create SQLite Database

In [27]:
import sqlite3

conn = sqlite3.connect('etl-project/data/final/master.db')

## Load Cleaned Tables

In [28]:
customers.to_sql("customers", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)

32951

In [30]:
import json
engagement['products'] = engagement['products'].apply(json.dumps)
engagement.to_sql("engagement", conn, if_exists="replace", index=False)

7

## Join Tables

In [31]:
query = """
SELECT
    e.id AS engagement_id,
    e.userId AS customer_id,
    p.product_id,
    p.product_category_name,
    c.customer_city,
    c.customer_state
FROM engagement e
LEFT JOIN customers c ON e.userId = c.customer_id
LEFT JOIN products p ON p.product_id = e.product_id;
"""

# Export Final Dataset

In [32]:
query = """SELECT e.id AS engagement_id, CAST(e.userid AS TEXT) AS customer_id, p.product_id, p.product_category_name, c.customer_city, c.customer_state FROM engagement e LEFT JOIN customers c ON CAST(e.userid AS TEXT) = c.customer_id LEFT JOIN json_each(e.products) AS engagement_product_item LEFT JOIN products p ON p.product_id = CAST(json_extract(engagement_product_item.value, '$.productId') AS TEXT);"""
master = pd.read_sql_query(query, conn)
master.to_csv("etl-project/data/final/master_dataset.csv", index=False)